# Chapter 6: The Training Loop

This notebook covers implementing training loops and optimization techniques.

## Training Arguments Configuration

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=500,
    fp16=True,
)

print("Training Arguments Configured")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

## Learning Rate Schedule Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulate cosine schedule
total_steps = 1000
warmup_steps = 100
learning_rate = 2e-4

steps = np.arange(total_steps)
lr_schedule = []

for step in steps:
    if step < warmup_steps:
        lr = learning_rate * step / warmup_steps
    else:
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        lr = learning_rate * 0.5 * (1 + np.cos(np.pi * progress))
    lr_schedule.append(lr)

plt.figure(figsize=(10, 5))
plt.plot(steps, lr_schedule)
plt.xlabel('Training Steps')
plt.ylabel('Learning Rate')
plt.title('Cosine Learning Rate Schedule')
plt.grid(True)
plt.show()

## Gradient Accumulation

In [ ]:
# Calculate effective batch size
per_device_batch_size = 4
gradient_accumulation_steps = 4
num_gpus = 1

effective_batch_size = per_device_batch_size * gradient_accumulation_steps * num_gpus

print(f"Per-device batch size: {per_device_batch_size}")
print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
print(f"Number of GPUs: {num_gpus}")
print(f"Effective batch size: {effective_batch_size}")

## Mixed Precision Training

In [ ]:
import torch

# Memory comparison
model_size = 7  # 7B parameters

fp32_memory = model_size * 4  # 4 bytes per parameter
fp16_memory = model_size * 2  # 2 bytes per parameter

print(f"FP32 memory: {fp32_memory:.2f} GB")
print(f"FP16 memory: {fp16_memory:.2f} GB")
print(f"Memory reduction: {(1 - fp16_memory/fp32_memory) * 100:.0f}%")

## Key Takeaways

1. Training loops require careful configuration
2. Learning rate scheduling is crucial for convergence
3. Gradient accumulation enables larger effective batch sizes
4. Mixed precision reduces memory and speeds up training
5. Checkpoint management saves progress